# Merge the Patch 16.19 Final Dataset

This notebook merges the Riot OTP observations with the Global D2+ LoLalytics matchup table. It preserves every Riot row, flags unavailable and low-sample matchups, checks that the merge creates no duplicate observations, and exports the combined dataset.

In [ ]:
%pip install -q pandas pyarrow

In [ ]:
from pathlib import Path
from datetime import date
import re
import pandas as pd

TARGET_PATCH = '16.19'
TARGET_RANK = 'd2_plus'
TARGET_REGION = 'global'
RECOMMENDED_MIN_GAMES = 50  # Flag only; no rows are deleted from the full export

start_dir = Path.cwd().resolve()
search_dirs = [start_dir, *start_dir.parents]
PROJECT_DIR = next(
    (p for p in search_dirs if (p / 'data' / 'processed').exists()),
    None,
)
if PROJECT_DIR is None:
    raise FileNotFoundError('Could not find the repository data/processed directory.')

PROCESSED_DIR = PROJECT_DIR / 'data' / 'processed'
print('Project directory:', PROJECT_DIR)

In [ ]:
def load_preferred(stem):
    parquet_path = PROCESSED_DIR / f'{stem}.parquet'
    csv_path = PROCESSED_DIR / f'{stem}.csv'
    if parquet_path.exists():
        return pd.read_parquet(parquet_path)
    if csv_path.exists():
        return pd.read_csv(csv_path)
    raise FileNotFoundError(f'Could not find {parquet_path} or {csv_path}.')

otp = load_preferred('otp_matches_16_19')
matchups = load_preferred('expected_matchups_16_19')

print(f'OTP observations: {len(otp):,}')
print(f'Expected-matchup rows: {len(matchups):,}')
display(matchups['status'].value_counts(dropna=False).rename('rows').to_frame())

## Prepare a one-row-per-matchup lookup

For a dated LoLalytics refresh, the merge checks that **every matchup needed by the current Riot observations** belongs to the same complete snapshot. An interrupted refresh must be resumed before the analysis files are overwritten. Older preliminary exports without dated snapshots still work.


In [ ]:
KEY = ['lane', 'champion', 'opponent_champion']
required_otp = {'match_id', 'puuid', *KEY}
required_matchups = {
    'patch', 'rank', 'region', 'status', 'expected_winrate', 'matchup_games', *KEY
}
missing_otp = required_otp.difference(otp.columns)
missing_matchups = required_matchups.difference(matchups.columns)
if missing_otp:
    raise ValueError(f'OTP table is missing columns: {sorted(missing_otp)}')
if missing_matchups:
    raise ValueError(f'Matchup table is missing columns: {sorted(missing_matchups)}')

lookup = matchups.copy()
lookup['patch'] = lookup['patch'].astype(str)
lookup = lookup.loc[
    lookup['patch'].eq(TARGET_PATCH)
    & lookup['rank'].eq(TARGET_RANK)
    & lookup['region'].eq(TARGET_REGION)
].copy()
lookup['expected_winrate'] = pd.to_numeric(lookup['expected_winrate'], errors='coerce')
lookup['matchup_games'] = pd.to_numeric(lookup['matchup_games'], errors='coerce')

if lookup.duplicated(KEY).any():
    duplicates = lookup.loc[lookup.duplicated(KEY, keep=False), KEY + ['status']]
    raise ValueError(f'Duplicate matchup keys found:\n{duplicates.head(20).to_string(index=False)}')

# A partial dated refresh would mix early-patch and later matchup estimates.
# Limit this check to keys actually used by the current OTP collection.
if 'snapshot_id' in lookup.columns:
    required_key_rows = otp[KEY].drop_duplicates().merge(
        lookup[KEY + ['snapshot_id', 'status']],
        on=KEY, how='left', validate='one_to_one', indicator=True,
    )
    required_key_rows['snapshot_id'] = (
        required_key_rows['snapshot_id'].astype('string').str.strip().replace('', pd.NA)
    )
    dated_ids = sorted(required_key_rows['snapshot_id'].dropna().unique().tolist())
    if dated_ids:
        def is_calendar_date(value):
            if not re.fullmatch(r'\d{4}-\d{2}-\d{2}', value):
                return False
            try:
                return date.fromisoformat(value).isoformat() == value
            except ValueError:
                return False

        expected_id = dated_ids[0]
        missing_keys = required_key_rows['_merge'].ne('both')
        mixed_or_legacy = required_key_rows['snapshot_id'].ne(expected_id).fillna(True)
        unfinished = ~required_key_rows['status'].isin(['ok', 'no_data'])
        if (len(dated_ids) != 1 or not is_calendar_date(expected_id)
                or missing_keys.any() or mixed_or_legacy.any() or unfinished.any()):
            raise ValueError(
                'LoLalytics snapshot is incomplete for this OTP collection: '
                f'{len(dated_ids)} distinct dated IDs {dated_ids}, '
                f'{int(missing_keys.sum())} required keys absent, '
                f'{int(mixed_or_legacy.sum())} keys on a different or legacy snapshot, '
                f'{int(unfinished.sum())} keys without ok/no_data status. '
                'Resume lolalytics_matchup_collection.ipynb with the SAME '
                'REFRESH_SNAPSHOT date until all required matchups complete, '
                'then rerun this merge notebook. Keep the checkpoint.'
            )
        print(f'Validated LoLalytics snapshot {expected_id} '
              f'for all {len(required_key_rows):,} required matchup keys.')

lookup = lookup.rename(columns={
    'rank': 'matchup_rank',
    'region': 'matchup_region',
    'status': 'matchup_status',
    'patch': 'matchup_patch',
})
keep_optional = [
    'raw_matchup_winrate', 'tier_average_winrate', 'lolalytics_delta2',
    'source_url', 'error', 'scraped_at_utc'
]
lookup_columns = KEY + [
    'matchup_patch', 'matchup_rank', 'matchup_region', 'matchup_status',
    'expected_winrate', 'matchup_games'
] + [column for column in keep_optional if column in lookup.columns]
lookup = lookup.loc[:, lookup_columns]
print(f'Validated lookup rows: {len(lookup):,}')

## Merge without dropping observations

In [ ]:
original_rows = len(otp)
merged = otp.merge(lookup, on=KEY, how='left', validate='many_to_one')

if len(merged) != original_rows:
    raise AssertionError('The merge changed the number of OTP observations.')
if merged.duplicated(['match_id', 'puuid']).any():
    raise AssertionError('The merge created duplicate seed-player match rows.')

merged['expected_winrate_prob'] = merged['expected_winrate'] / 100
merged['matchup_data_available'] = merged['matchup_status'].eq('ok')
for threshold in (10, 25, 50, 100, 200):
    merged[f'matchup_games_ge_{threshold}'] = (
        merged['matchup_data_available'] & merged['matchup_games'].ge(threshold)
    )
merged['matchup_reliable'] = merged[f'matchup_games_ge_{RECOMMENDED_MIN_GAMES}']

print(f'Merged observations: {len(merged):,}')
print(f'Unique Riot matches: {merged["match_id"].nunique():,}')
print(f'Rows with successful matchup data: {merged["matchup_data_available"].sum():,} '
      f'({merged["matchup_data_available"].mean():.1%})')

## Compare possible minimum-game cutoffs

This reports how many player observations and unique Riot matches each cutoff would retain. The full export remains unfiltered.

In [ ]:
summary_rows = []
for threshold in (0, 10, 25, 50, 100, 200):
    mask = merged["matchup_data_available"].copy()
    if threshold:
        mask &= merged['matchup_games'].ge(threshold)
    summary_rows.append({
        'minimum_matchup_games': threshold,
        'player_rows_retained': int(mask.sum()),
        'player_rows_retained_pct': float(mask.mean()),
        'unique_matches_retained': int(merged.loc[mask, 'match_id'].nunique()),
        'unique_matchup_keys_retained': int(merged.loc[mask, KEY].drop_duplicates().shape[0]),
    })
quality_summary = pd.DataFrame(summary_rows)
display(quality_summary.style.format({'player_rows_retained_pct': '{:.1%}'}))

unresolved = (
    merged.loc[~merged['matchup_data_available'], KEY + ['matchup_status']]
    .drop_duplicates()
    .sort_values(KEY)
    .reset_index(drop=True)
)
print(f'Unresolved matchup keys represented in the OTP data: {len(unresolved):,}')
display(unresolved.head(20))

## Save the combined dataset

In [ ]:
output_stem = 'otp_analysis_16_19'
csv_path = PROCESSED_DIR / f'{output_stem}.csv'
parquet_path = PROCESSED_DIR / f'{output_stem}.parquet'
summary_path = PROCESSED_DIR / 'matchup_quality_summary_16_19.csv'
unresolved_path = PROCESSED_DIR / 'unresolved_matchups_16_19.csv'

merged.to_csv(csv_path, index=False)
merged.to_parquet(parquet_path, index=False)
quality_summary.to_csv(summary_path, index=False)
unresolved.to_csv(unresolved_path, index=False)

print('Final CSV:', csv_path)
print('Final Parquet:', parquet_path)
print('Cutoff comparison:', summary_path)
print('Unresolved matchups:', unresolved_path)

In [ ]:
model_df = merged.loc[merged["matchup_games_ge_50"]].copy()

model_df.to_csv(
    PROCESSED_DIR / "otp_analysis_16_19_min50.csv",
    index=False,
)

model_df.to_parquet(
    PROCESSED_DIR / "otp_analysis_16_19_min50.parquet",
    index=False,
)

print(model_df.shape)